# ⛓️ LangChain Chains — Complete Tutorial
### *Connect LLM steps like LEGO blocks*

---

> **LangChain version:** 1.3.6 | **Python:** 3.11  
> **Prerequisites:** OpenAI API key, basic Python knowledge

### 🗺️ What We'll Cover

| # | Concept | What You'll Build |
|---|---------|-------------------|
| 1 | **Why Chains?** | The problem they solve |
| 2 | **The `\|` Pipe Operator** | Your first chain |
| 3 | **Output Parsers** | Clean, usable output |
| 4 | **Sequential Chains** | Output of A → Input of B |
| 5 | **Parallel Chains** | Multiple chains at once |
| 6 | **Branching Chains** | Different paths based on input |
| 7 | **Real-world Mini Project** | Movie review analyzer |

---
*Every cell is self-contained and runnable. Code is kept simple — the goal is understanding the concept, not complexity.*

---
## 📦 Setup

Run once to install and connect to the LLM.

In [ ]:
!pip install -q langchain langchain-openai langchain-community openai

In [1]:
import sys, requests
import importlib.metadata
import openai

print("Python:", sys.version)
print("Requests:", requests.__version__)
print("OpenAI:", openai.__version__)
print("langchain:", importlib.metadata.version("langchain"))
print("langchain-community:", importlib.metadata.version("langchain-community"))
print("langchain-openai:", importlib.metadata.version("langchain-openai"))

Python: 3.11.9 (tags/v3.11.9:de54cf5, Apr  2 2024, 10:12:12) [MSC v.1938 64 bit (AMD64)]
Requests: 2.33.1
OpenAI: 2.36.0
langchain: 1.3.6
langchain-community: 0.4.2
langchain-openai: 1.2.2


In [ ]:
import os
from langchain_openai import ChatOpenAI

os.environ["OPENAI_API_KEY"] = "sk-..."   # ← paste your key here

llm = ChatOpenAI(model="gpt-3.5-turbo", temperature=0.7)

# Quick check
response = llm.invoke("Say hello in one word.")
print("✅ Connected! Test response:", response.content)

---
## 1️⃣ Why Do We Need Chains?

### The Problem

Imagine you want to:
1. Take a topic from the user
2. Generate a blog post about it
3. Translate that blog post to French

**Without chains**, you'd write this:

```python
# ❌ Manual, messy, hard to reuse
response1 = llm.invoke(f"Write a blog post about {topic}")
blog_post  = response1.content

response2  = llm.invoke(f"Translate this to French: {blog_post}")
translated = response2.content
```

**With chains**, the same thing becomes:

```python
# ✅ Clean, readable, reusable
result = (write_chain | translate_chain).invoke({"topic": topic})
```

### What is a Chain?

> A **Chain** is a sequence of steps connected together, where the **output of one step becomes the input of the next**.

```
Input
  ↓
Step 1: Prompt Template   (formats your input)
  ↓
Step 2: LLM               (generates a response)
  ↓
Step 3: Output Parser     (cleans the response)
  ↓
Output
```

LangChain uses the `|` pipe operator to connect steps — just like Unix terminal pipes!

In [ ]:
# Let's see the PROBLEM first — manual, no chains
from langchain_openai import ChatOpenAI

llm = ChatOpenAI(model="gpt-3.5-turbo", temperature=0.7)

topic = "black holes"

# Step 1: manual call
response1 = llm.invoke(f"Give one interesting fact about {topic}.")
fact = response1.content

# Step 2: manual call — using output of step 1
response2 = llm.invoke(f"Make this fact sound more exciting: {fact}")
exciting_fact = response2.content

print("Original fact:")
print(fact)
print("\nExciting version:")
print(exciting_fact)
print("\n⚠️  Works, but imagine doing this for 10 steps...")

---
## 2️⃣ The `|` Pipe Operator — Your First Chain

### LCEL: LangChain Expression Language

LangChain introduced **LCEL** (LangChain Expression Language) — a clean way to build chains using the `|` pipe.

```
prompt | llm | parser
```

Reads left to right:
- `prompt` formats the input into a proper message
- `llm` sends it to the model and gets a response  
- `parser` converts the response into a clean string

### The 3 Core Building Blocks

| Block | Import | What it does |
|-------|--------|-------------|
| `ChatPromptTemplate` | `langchain_core.prompts` | Formats your input with variables |
| `ChatOpenAI` | `langchain_openai` | Calls the LLM |
| `StrOutputParser` | `langchain_core.output_parsers` | Extracts just the text string |

In [ ]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

# ── Block 1: Prompt Template ───────────────────────────────────────
# {topic} is a variable — gets filled in when you invoke the chain
prompt = ChatPromptTemplate.from_template(
    "Give one interesting fact about {topic} in exactly 2 sentences."
)

# See what the prompt looks like BEFORE sending to LLM
rendered = prompt.format_messages(topic="black holes")
print("Rendered prompt:")
for msg in rendered:
    print(f"  [{msg.type}]: {msg.content}")

In [ ]:
# ── Block 2: The Parser ───────────────────────────────────────────
parser = StrOutputParser()

# Without parser: response is an AIMessage object
raw_response = llm.invoke(rendered)
print("Without parser:")
print(f"  Type: {type(raw_response)}")
print(f"  Value: {raw_response}")

print()

# With parser: response is a plain string
clean_response = parser.invoke(raw_response)
print("With parser:")
print(f"  Type: {type(clean_response)}")
print(f"  Value: {clean_response}")

In [ ]:
# ── Block 3: Connect with | to make a Chain ──────────────────────
# This is the magic moment — three objects connected with |
chain = prompt | llm | StrOutputParser()

print("Chain type:", type(chain))
print()

# Invoke the chain — just pass the variable values
result = chain.invoke({"topic": "black holes"})

print("Result type:", type(result))   # plain string!
print("Result:", result)

In [ ]:
# ── Reusability — same chain, different inputs ───────────────────
topics = ["quantum computers", "the Great Wall of China", "the human brain", "volcanoes"]

print("🌍 Interesting Facts:\n")
for topic in topics:
    fact = chain.invoke({"topic": topic})
    print(f"🔷 {topic.title()}:")
    print(f"   {fact}")
    print()

In [ ]:
# ── Multiple variables in a prompt ──────────────────────────────
# Prompts can have as many {variables} as you need

explain_chain = (
    ChatPromptTemplate.from_template(
        "Explain {concept} to someone who is {audience}. "
        "Keep it under {sentences} sentences."
    )
    | llm
    | StrOutputParser()
)

# Invoke with all three variables
result = explain_chain.invoke({
    "concept": "machine learning",
    "audience": "a 12-year-old",
    "sentences": 3
})
print(result)

In [ ]:
# ✏️ YOUR TURN
# Change the concept, audience, and sentences below and run!

result = explain_chain.invoke({
    "concept":   "the stock market",     # ← change this
    "audience":  "a curious grandparent", # ← change this
    "sentences": 3                         # ← change this
})
print(result)

---
## 3️⃣ Output Parsers — Getting Useful Data Back

The LLM always returns raw text. **Output parsers** convert that text into something your code can actually use.

| Parser | Output type | Use when you need |
|--------|------------|-------------------|
| `StrOutputParser` | `str` | Plain text — most common |
| `JsonOutputParser` | `dict` | Structured data |
| `CommaSeparatedListOutputParser` | `list` | A list of items |
| `PydanticOutputParser` | Pydantic model | Typed, validated data |

In [ ]:
from langchain_core.output_parsers import StrOutputParser, JsonOutputParser
from langchain_core.output_parsers import CommaSeparatedListOutputParser

# ── Parser 1: StrOutputParser (you've seen this) ──────────────────
str_chain = (
    ChatPromptTemplate.from_template("Name the capital of {country}.")
    | llm
    | StrOutputParser()
)
result = str_chain.invoke({"country": "Japan"})
print(f"StrOutputParser → type: {type(result).__name__} | value: {result}")

In [ ]:
# ── Parser 2: CommaSeparatedListOutputParser ──────────────────────
list_parser = CommaSeparatedListOutputParser()

list_chain = (
    ChatPromptTemplate.from_template(
        "List 5 {category}. Return ONLY a comma-separated list, nothing else."
    )
    | llm
    | list_parser
)

result = list_chain.invoke({"category": "programming languages"})
print(f"Type: {type(result).__name__}")
print(f"Value: {result}")
print(f"First item: {result[0]}")
print(f"Count: {len(result)}")

In [ ]:
# ── Parser 3: JsonOutputParser ────────────────────────────────────
# Tell the LLM exactly what JSON shape you want back

json_chain = (
    ChatPromptTemplate.from_template(
        """Return information about {city} as JSON with these exact keys:
        name, country, population_millions, famous_for (list of 3 things).
        Return ONLY valid JSON, no extra text."""
    )
    | llm
    | JsonOutputParser()
)

result = json_chain.invoke({"city": "Tokyo"})

print(f"Type: {type(result).__name__}")
print(f"City: {result['name']}")
print(f"Country: {result['country']}")
print(f"Population: {result['population_millions']}M")
print(f"Famous for: {result['famous_for']}")

In [ ]:
# ── Parser 4: Pydantic — typed and validated output ──────────────
from langchain_core.output_parsers import JsonOutputParser
from pydantic import BaseModel, Field
from typing import List

# Define the exact shape of data you want
class MovieReview(BaseModel):
    title: str = Field(description="Movie title")
    year: int = Field(description="Release year")  
    rating: float = Field(description="Rating out of 10")
    pros: List[str] = Field(description="List of 2 things done well")
    cons: List[str] = Field(description="List of 2 weaknesses")
    verdict: str = Field(description="One sentence summary")

parser = JsonOutputParser(pydantic_object=MovieReview)

review_chain = (
    ChatPromptTemplate.from_messages([
        ("system", "You are a film critic. Always respond in valid JSON."),
        ("human", "Review the movie '{title}'. {instructions}"),
    ])
    | llm
    | parser
)

result = review_chain.invoke({
    "title": "Inception",
    "instructions": parser.get_format_instructions()
})

print(f"🎬 {result['title']} ({result['year']}) — ⭐ {result['rating']}/10")
print(f"✅ Pros: {result['pros']}")
print(f"❌ Cons: {result['cons']}")
print(f"📝 Verdict: {result['verdict']}")

---
## 4️⃣ Sequential Chains — Chaining LLM Calls Together

This is where chains get powerful. The **output of one LLM call becomes the input of the next**.

```
Topic
  ↓
Chain A: "Write a blog post about {topic}"
  ↓  (blog post text)
Chain B: "Summarize this into 3 bullet points: {blog_post}"
  ↓  (bullet points)
Chain C: "Translate these bullet points to Spanish: {bullets}"
  ↓
Final Output
```

LangChain uses `RunnablePassthrough` to route values between steps.

In [ ]:
from langchain_core.runnables import RunnablePassthrough

# ── Simple 2-step sequential chain ───────────────────────────────
# Step 1: Write a poem
write_poem = (
    ChatPromptTemplate.from_template("Write a short 4-line poem about {topic}.")
    | llm
    | StrOutputParser()
)

# Step 2: Analyse the poem (receives poem as input)
analyse_poem = (
    ChatPromptTemplate.from_template(
        "Analyse this poem in one sentence — identify its mood and style:\n\n{poem}"
    )
    | llm
    | StrOutputParser()
)

# Connect them: output of write_poem becomes "poem" in analyse_poem
sequential = (
    {"poem": write_poem}          # write_poem runs first, result stored as "poem"
    | RunnablePassthrough.assign(analysis=analyse_poem)  # analyse_poem runs second
)

result = sequential.invoke({"topic": "autumn leaves"})

print("📜 POEM:")
print(result["poem"])
print("\n🔍 ANALYSIS:")
print(result["analysis"])

In [ ]:
# ── 3-step pipeline: generate → improve → translate ──────────────

# Step 1: Generate a joke
generate_joke = (
    ChatPromptTemplate.from_template("Write a short, clean joke about {topic}.")
    | llm | StrOutputParser()
)

# Step 2: Improve the joke (uses output of step 1)
improve_joke = (
    ChatPromptTemplate.from_template(
        "Make this joke funnier by adding a punchline twist:\n{joke}"
    )
    | llm | StrOutputParser()
)

# Step 3: Rate the improved joke (uses output of step 2)
rate_joke = (
    ChatPromptTemplate.from_template(
        """Rate this joke on a scale of 1-10 for funniness and explain why in one sentence:
        
{improved_joke}"""
    )
    | llm | StrOutputParser()
)

# Wire them together
pipeline = (
    {"joke": generate_joke, "topic": RunnablePassthrough()}
    | RunnablePassthrough.assign(improved_joke=lambda x: improve_joke.invoke({"joke": x["joke"]}))
    | RunnablePassthrough.assign(rating=lambda x: rate_joke.invoke({"improved_joke": x["improved_joke"]}))
)

result = pipeline.invoke({"topic": "programmers"})

print("😄 ORIGINAL JOKE:")
print(result["joke"])
print("\n✨ IMPROVED JOKE:")
print(result["improved_joke"])
print("\n⭐ RATING:")
print(result["rating"])

In [ ]:
# ── Real use case: Study notes pipeline ──────────────────────────
# Input: a topic → Output: explanation + quiz question + memory tip

# Chain 1: Explain the topic
explain = (
    ChatPromptTemplate.from_template(
        "Explain {topic} clearly in 3 bullet points for a student."
    )
    | llm | StrOutputParser()
)

# Chain 2: Create a quiz question from the explanation
make_quiz = (
    ChatPromptTemplate.from_template(
        "Based on this explanation:\n{explanation}\n\n"
        "Write ONE multiple choice question (A/B/C/D) with the answer marked."
    )
    | llm | StrOutputParser()
)

# Chain 3: Create a memory trick from the explanation
memory_trick = (
    ChatPromptTemplate.from_template(
        "Based on this explanation:\n{explanation}\n\n"
        "Give ONE creative memory trick or acronym to remember this."
    )
    | llm | StrOutputParser()
)

# Full pipeline
study_pipeline = (
    {"explanation": explain, "topic": RunnablePassthrough()}
    | RunnablePassthrough.assign(quiz=lambda x: make_quiz.invoke({"explanation": x["explanation"]}))
    | RunnablePassthrough.assign(tip=lambda x: memory_trick.invoke({"explanation": x["explanation"]}))
)

result = study_pipeline.invoke({"topic": "photosynthesis"})

print("📚 EXPLANATION:")
print(result["explanation"])
print("\n❓ QUIZ QUESTION:")
print(result["quiz"])
print("\n🧠 MEMORY TRICK:")
print(result["tip"])

---
## 5️⃣ Parallel Chains — Run Multiple Chains Simultaneously

Sometimes you don't need the output of one chain to feed into another.  
You just want to **run several chains on the same input at the same time**.

`RunnableParallel` does exactly that — all chains run concurrently, saving time.

```
              ┌─── Chain A: pros ────┐
Input ────────┼─── Chain B: cons ────┼──→ Combined Result
              └─── Chain C: summary ─┘
```

In [ ]:
from langchain_core.runnables import RunnableParallel

# Three chains that all take {topic} as input
pros_chain = (
    ChatPromptTemplate.from_template("List 3 advantages of {topic}. Be concise.")
    | llm | StrOutputParser()
)

cons_chain = (
    ChatPromptTemplate.from_template("List 3 disadvantages of {topic}. Be concise.")
    | llm | StrOutputParser()
)

verdict_chain = (
    ChatPromptTemplate.from_template(
        "Give a one-sentence balanced verdict on {topic}: is it worth it overall?"
    )
    | llm | StrOutputParser()
)

# Run all three at the same time
parallel = RunnableParallel(
    pros=pros_chain,
    cons=cons_chain,
    verdict=verdict_chain
)

result = parallel.invoke({"topic": "remote work"})

print("✅ PROS:")
print(result["pros"])
print("\n❌ CONS:")
print(result["cons"])
print("\n⚖️  VERDICT:")
print(result["verdict"])

In [ ]:
# ── Parallel saves time — let's measure it ────────────────────────
import time

topic = "electric vehicles"

# Sequential (one by one)
start = time.time()
p1 = pros_chain.invoke({"topic": topic})
c1 = cons_chain.invoke({"topic": topic})
v1 = verdict_chain.invoke({"topic": topic})
sequential_time = time.time() - start

# Parallel (all at once)
start = time.time()
result = parallel.invoke({"topic": topic})
parallel_time = time.time() - start

print(f"⏱️  Sequential: {sequential_time:.2f}s")
print(f"⚡ Parallel:   {parallel_time:.2f}s")
print(f"🚀 Speedup:    {sequential_time/parallel_time:.1f}x faster")

In [ ]:
# ── Parallel + Sequential combined ────────────────────────────────
# First run parallel chains, then feed combined results into a final chain

summary_chain = (
    ChatPromptTemplate.from_template(
        """You have analysed {topic}. Here is the analysis:

PROS: {pros}
CONS: {cons}  
VERDICT: {verdict}

Write a 3-sentence recommendation for whether someone should adopt {topic}."""
    )
    | llm | StrOutputParser()
)

# Step 1: parallel analysis → Step 2: final recommendation
full_analysis = parallel | RunnablePassthrough.assign(
    recommendation=lambda x: summary_chain.invoke({
        "topic": topic,
        "pros": x["pros"],
        "cons": x["cons"],
        "verdict": x["verdict"]
    })
)

result = full_analysis.invoke({"topic": "social media for teenagers"})

print("📋 FINAL RECOMMENDATION:")
print(result["recommendation"])

---
## 6️⃣ Branching Chains — Different Paths Based on Input

Sometimes you want the chain to **take different routes** depending on the input.  
This is called a **Router Chain**.

```
Input: "What is 2+2?"       → Route: MATH   → Math explanation chain
Input: "Why is sky blue?"   → Route: SCIENCE → Science explanation chain  
Input: "Who wrote Hamlet?"  → Route: HISTORY → History explanation chain
```

We use `RunnableLambda` to write custom routing logic in plain Python.

In [ ]:
from langchain_core.runnables import RunnableLambda

# ── Define specialist chains ──────────────────────────────────────
math_chain = (
    ChatPromptTemplate.from_template(
        "You are a math tutor. Answer this clearly with steps: {question}"
    )
    | llm | StrOutputParser()
)

science_chain = (
    ChatPromptTemplate.from_template(
        "You are a science teacher. Explain this with a real-world example: {question}"
    )
    | llm | StrOutputParser()
)

history_chain = (
    ChatPromptTemplate.from_template(
        "You are a history professor. Answer this with historical context: {question}"
    )
    | llm | StrOutputParser()
)

general_chain = (
    ChatPromptTemplate.from_template(
        "You are a helpful assistant. Answer this question: {question}"
    )
    | llm | StrOutputParser()
)

# ── Step 1: Classify the question ────────────────────────────────
classify_chain = (
    ChatPromptTemplate.from_template(
        """Classify this question into exactly ONE category: math, science, history, or general.
        
Question: {question}
        
Reply with ONLY the category word, nothing else."""
    )
    | llm | StrOutputParser()
)

# ── Step 2: Route to the right chain ─────────────────────────────
def route(inputs: dict) -> str:
    category = classify_chain.invoke({"question": inputs["question"]}).strip().lower()
    print(f"   📂 Classified as: {category}")
    
    if "math" in category:
        return math_chain.invoke(inputs)
    elif "science" in category:
        return science_chain.invoke(inputs)
    elif "history" in category:
        return history_chain.invoke(inputs)
    else:
        return general_chain.invoke(inputs)

# ── Step 3: Wrap in a chain ────────────────────────────────────────
router_chain = RunnableLambda(route)

# Test it with different questions
questions = [
    "What is the quadratic formula?",
    "Why do leaves change colour in autumn?",
    "Who was the first President of the United States?",
    "What are some tips for better sleep?",
]

for q in questions:
    print(f"\n❓ Question: {q}")
    answer = router_chain.invoke({"question": q})
    print(f"💬 Answer: {answer[:200]}...")

---
## 7️⃣ Mini Project — Movie Review Analyzer

Let's put everything together into a **real, useful pipeline**.

### What it does:
1. Takes a movie title as input
2. **Parallel:** fetches plot summary + gets critical reception (two chains at once)
3. **Sequential:** uses both outputs to generate a structured review
4. **Structured output:** returns a typed Python object you can use in your app

This is exactly the kind of pipeline you'd build in a real product.

In [ ]:
from langchain_core.runnables import RunnableParallel, RunnablePassthrough
from langchain_core.output_parsers import JsonOutputParser
from pydantic import BaseModel, Field
from typing import List

# ── Output schema ─────────────────────────────────────────────────
class FullReview(BaseModel):
    title: str
    genre: List[str]
    one_line_summary: str
    what_works: List[str] = Field(description="3 things done well")
    what_doesnt: List[str] = Field(description="2 weaknesses")
    similar_movies: List[str] = Field(description="3 movies to watch if you liked this")
    score: float = Field(description="Score out of 10")
    recommended_for: str = Field(description="Who would enjoy this most")

parser = JsonOutputParser(pydantic_object=FullReview)

# ── Parallel chains ────────────────────────────────────────────────
plot_chain = (
    ChatPromptTemplate.from_template(
        "Write a 2-sentence spoiler-free plot summary of the movie '{title}'."
    )
    | llm | StrOutputParser()
)

reception_chain = (
    ChatPromptTemplate.from_template(
        "Describe in 2 sentences how '{title}' was received by critics and audiences."
    )
    | llm | StrOutputParser()
)

parallel_research = RunnableParallel(
    plot=plot_chain,
    reception=reception_chain,
    title=RunnablePassthrough()   # pass title through unchanged
)

# ── Final review chain ─────────────────────────────────────────────
final_review_chain = (
    ChatPromptTemplate.from_messages([
        ("system", "You are a film critic. Always respond in valid JSON only."),
        ("human", """Based on this research, write a full review of '{title}'.

Plot: {plot}
Critical Reception: {reception}

{format_instructions}"""),
    ])
    | llm
    | parser
)

# ── Full pipeline ──────────────────────────────────────────────────
def analyze_movie(title: str) -> dict:
    # Step 1: parallel research
    research = parallel_research.invoke({"title": title})
    
    # Step 2: final review using research
    review = final_review_chain.invoke({
        "title": title,
        "plot": research["plot"],
        "reception": research["reception"],
        "format_instructions": parser.get_format_instructions()
    })
    return review

# Run it!
review = analyze_movie("The Dark Knight")

print(f"🎬 {review['title']}")
print(f"🎭 Genre: {', '.join(review['genre'])}")
print(f"⭐ Score: {review['score']}/10")
print(f"\n📖 Summary: {review['one_line_summary']}")
print(f"\n✅ What Works:")
for w in review['what_works']:
    print(f"   • {w}")
print(f"\n❌ What Doesn't:")
for w in review['what_doesnt']:
    print(f"   • {w}")
print(f"\n🎯 Recommended For: {review['recommended_for']}")
print(f"\n🎞️  Watch Next: {', '.join(review['similar_movies'])}")

In [ ]:
# ✏️ Try with any movie!
review = analyze_movie("Parasite")  # ← change this

print(f"🎬 {review['title']} | ⭐ {review['score']}/10")
print(f"📖 {review['one_line_summary']}")

---
## 📖 Summary — Everything We Covered

```python
# ── 1. BASIC CHAIN ──────────────────────────────────────────────
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

chain = ChatPromptTemplate.from_template("...{var}...") | llm | StrOutputParser()
result = chain.invoke({"var": "value"})   # result is a plain string

# ── 2. OUTPUT PARSERS ───────────────────────────────────────────
StrOutputParser()                         # → str
CommaSeparatedListOutputParser()          # → list
JsonOutputParser(pydantic_object=MyModel) # → dict (typed)

# ── 3. SEQUENTIAL CHAIN ─────────────────────────────────────────
from langchain_core.runnables import RunnablePassthrough

pipeline = (
    {"step1_output": chain_a, "original": RunnablePassthrough()}
    | RunnablePassthrough.assign(step2_output=chain_b)
)

# ── 4. PARALLEL CHAIN ───────────────────────────────────────────
from langchain_core.runnables import RunnableParallel

parallel = RunnableParallel(result_a=chain_a, result_b=chain_b, result_c=chain_c)
result = parallel.invoke({"input": "..."})
# result = {"result_a": "...", "result_b": "...", "result_c": "..."}

# ── 5. BRANCHING CHAIN ──────────────────────────────────────────
from langchain_core.runnables import RunnableLambda

def route(inputs):
    if condition:
        return chain_a.invoke(inputs)
    else:
        return chain_b.invoke(inputs)

router = RunnableLambda(route)
```

---

## 🔜 Coming Up Next

| Topic | What You'll Learn |
|-------|------------------|
| 🔧 **Tools** | Give chains the ability to search the web, run code, call APIs |
| 🧠 **Memory** | Make chains remember previous conversations |
| 🤖 **Agents** | Let the LLM decide WHICH chain/tool to use |
